# Lesson 4.4 — Agent Patterns and Frameworks

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Recognize common agent patterns (tool-use loop, subagents, memory)
- Describe manager/supervisor multi-agent orchestration
- Compare agentic frameworks: Strands, LangGraph, PydanticAI
- Pick a pattern/framework for a given multi-step task

**You will produce:** A manager/subagent orchestration demo plus a framework comparison reference.

## Concepts

Reusable **agent patterns** (seen across Phases 2-4):

- **Tool-use loop**: the core call->tool->result cycle (Lesson 4.2).
- **Subagents**: isolated workers for parallel/independent subtasks (Lesson 2.4).
- **Manager/supervisor**: one agent plans and delegates to subagents, then synthesizes.
- **Memory & context management**: compaction and pruning to stay within the window.

**Frameworks** provide abstractions so you do not hand-roll everything:
- **Strands**: Anthropic's agent framework/toolkit for building agents.
- **LangGraph**: graph-based orchestration of nodes/edges for stateful, multi-step agent workflows.
- **PydanticAI**: a Python agent framework emphasizing typed, validated inputs/outputs via Pydantic.

> Frameworks move fast - treat these as accurate summaries and confirm current APIs on each project's official docs before building.

> **Reference:** [Anthropic - Agents and tools](https://docs.anthropic.com/en/docs/agents-and-tools/overview)

In [ ]:
# !pip install anthropic
import os

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {prompt[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Manager/subagent orchestration

A manager delegates subtasks to subagents and synthesizes the results.

In [ ]:
def subagent(role, data):
    if not HAVE_API:
        return f"[MOCK {role}] {data[:30]}"
    msg = client.messages.create(model=MODEL_HAIKU, max_tokens=150,
        system=role, messages=[{"role": "user", "content": data}])
    return msg.content[0].text

def manager(topic):
    research = subagent("You research facts. Be terse.", f"Key facts about {topic}")
    risks = subagent("You list risks. Be terse.", f"Risks of {topic}")
    synthesis = subagent("You synthesize a brief.",
        f"Facts: {research}\nRisks: {risks}\nWrite a 2-line brief.")
    return synthesis

print(manager("adopting prompt caching"))

## Code-Along: Step 2 — Framework comparison table

A quick reference for choosing an abstraction.

In [ ]:
frameworks = [
    ("Strands", "Anthropic agent toolkit", "Claude-native agent building"),
    ("LangGraph", "graph of nodes/edges, stateful", "complex branching workflows"),
    ("PydanticAI", "typed, validated I/O via Pydantic", "type-safe structured agents"),
]
for name, model, best in frameworks:
    print(f"{name:12} | {model:32} | best for: {best}")

## Exercise

Write `pick_framework(task)` returning 'Strands', 'LangGraph', or 'PydanticAI' with a comment justifying each branch (e.g., strict typing -> PydanticAI; complex branching -> LangGraph).

In [ ]:
def pick_framework(task):
    # TODO: if task.get('strict_types'): return 'PydanticAI'
    # TODO: if task.get('complex_branching'): return 'LangGraph'
    # TODO: else return 'Strands'
    pass

## Key Takeaways

- Core patterns: tool-use loop, subagents, manager/supervisor, memory management.
- Managers plan and delegate to subagents, then synthesize results.
- Strands (Claude-native), LangGraph (graph orchestration), PydanticAI (typed I/O).
- Confirm framework APIs on their official docs - they change frequently.

---

**Next:** [Lesson 4.5 — Operating Claude Code: CLAUDE.md, settings.json, and Slash Commands](./lesson-4.5-claude-code-operation.ipynb)